In [1]:
import requests
import pandas as pd
from IPython.display import display, HTML
API_KEY = "YOUR_TMDB_API_KEY"
BASE_URL = "https://api.themoviedb.org/3"
response = requests.get(
    f"{BASE_URL}/movie/popular",
    params={"api_key": API_KEY, "language": "en-US", "page": 1},
    timeout=20
)

print("Status Code:", response.status_code)
print(response.text[:500])


Status Code: 200
{"page":1,"results":[{"adult":false,"backdrop_path":"/1x9e0qWonw634NhIsRdvnneeqvN.jpg","genre_ids":[10749,18],"id":1523145,"original_language":"ru","original_title":"Твоё сердце будет разбито","overview":"High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple develops real feelings, but her family and classmates have reasons


In [2]:
def fetch_json(endpoint, params=None):
    if params is None:
        params = {}

    params["api_key"] = API_KEY
    url = f"{BASE_URL}{endpoint}"

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return response.json()


all_movies = []

for page in range(1, 6):
    data = fetch_json("/movie/popular", {"language": "en-US", "page": page})
    all_movies.extend(data.get("results", []))

df = pd.DataFrame(all_movies)

print("Total movies fetched:", len(df))
display(df.head())

Total movies fetched: 100


,adult,backdrop_path,genre_ids,id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count
0,False,/1x9e0qWonw634NhIsRdvnneeqvN.jpg,"[10749, 18]",1523145,ru,Твоё сердце будет разбито,High school student Polina is saved from bully...,843.1812,/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg,2026-03-26,Your Heart Will Be Broken,False,6.333,15
1,False,/uNToXatdunyvWXyXMrTI1nLvh6r.jpg,"[35, 878, 80]",1115544,en,Mike & Nick & Nick & Alice,Two gangsters and the woman they love try to s...,358.5571,/7F0jc75HrSkLVcvOXR2FXAIwuEv.jpg,2026-03-14,Mike & Nick & Nick & Alice,False,6.564,47
2,False,/6GuqUJ33BnWDAoVPZP55gDr5G55.jpg,"[28, 53, 27]",1084187,en,Pretty Lethal,A troupe of ballerinas find themselves fightin...,308.4196,/znTPnXCK3lEQJgqXCvP7e5FUz6f.jpg,2026-03-13,Pretty Lethal,False,6.800,114
3,False,/u8DU5fkLoM5tTRukzPC31oGPxaQ.jpg,"[878, 12, 14]",83533,en,Avatar: Fire and Ash,In the wake of the devastating war against the...,307.5829,/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg,2025-12-17,Avatar: Fire and Ash,False,7.259,1961
4,False,/afHAM9qOTmrVDeaLIU7zfssotUY.jpg,"[878, 12]",687163,en,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,307.2809,/yihdXomYb5kTeSivtFndMy5iDmf.jpg,2026-03-15,Project Hail Mary,False,8.153,681


In [4]:
genre_data = fetch_json("/genre/movie/list", {"language": "en-US"})

genre_map = {g["id"]: g["name"] for g in genre_data["genres"]}

print("Genre map created successfully")
print(genre_map)

Genre map created successfully
{28: 'Action', 12: 'Adventure', 16: 'Animation', 35: 'Comedy', 80: 'Crime', 99: 'Documentary', 18: 'Drama', 10751: 'Family', 14: 'Fantasy', 36: 'History', 27: 'Horror', 10402: 'Music', 9648: 'Mystery', 10749: 'Romance', 878: 'Science Fiction', 10770: 'TV Movie', 53: 'Thriller', 10752: 'War', 37: 'Western'}


In [12]:
df = df[["id", "title", "overview", "genre_ids", "release_date", "vote_average", "popularity"]].copy()

df.drop_duplicates(subset=["id"], inplace=True)
df["title"] = df["title"].fillna("")
df["overview"] = df["overview"].fillna("")
df["vote_average"] = df["vote_average"].fillna(0)
df["popularity"] = df["popularity"].fillna(0)
df["release_date"] = df["release_date"].fillna("Unknown")

def get_genres(ids):
    if isinstance(ids, list):
        return [genre_map.get(i, "Unknown") for i in ids]
    return []

df["genres"] = df["genre_ids"].apply(get_genres)

df = df[df["title"] != ""]
df = df[df["overview"].str.len() > 20]
df = df[df["genres"].apply(lambda x: len(x) > 0)]

print("Cleaned dataset shape:", df.shape)
display(df.head())
df.to_csv("movies_cleaned.csv", index=False)
print("CSV saved successfully")
mood_to_genres = {
    "happy": ["Comedy", "Animation", "Family"],
    "sad": ["Drama", "Romance"],
    "excited": ["Action", "Adventure", "Thriller"],
    "relaxed": ["Fantasy", "Family", "Music"],
    "romantic": ["Romance", "Drama"],
    "scared": ["Horror", "Mystery", "Thriller"]
}
def mood_score(movie_genres, target_genres, rating, popularity):
    genre_matches = sum(1 for g in target_genres if g in movie_genres)
    return genre_matches * 3 + (rating / 2) + (popularity / 100)


def recommend_movies(user_mood, top_n=5):
    user_mood = user_mood.strip().lower()

    if user_mood not in mood_to_genres:
        print("Invalid mood. Choose from:", ", ".join(mood_to_genres.keys()))
        return pd.DataFrame()

    target_genres = mood_to_genres[user_mood]

    temp = df.copy()
    temp["mood_score"] = temp.apply(
        lambda row: mood_score(row["genres"], target_genres, row["vote_average"], row["popularity"]),
        axis=1
    )

    temp["matched_genres"] = temp["genres"].apply(
        lambda genres: [g for g in target_genres if g in genres]
    )

    temp = temp[temp["matched_genres"].apply(len) > 0]
    temp = temp.sort_values(by=["mood_score", "vote_average", "popularity"], ascending=False)

    return temp[["title", "genres", "matched_genres", "overview", "release_date", "vote_average", "mood_score"]].head(top_n)
def show_html_results(results, mood):
    if results.empty:
        display(HTML("<h3>No results found.</h3>"))
        return

    cards = ""
    for _, row in results.iterrows():
        cards += f"""
        <div class='card'>
            <h2>{row['title']}</h2>
            <p><b>Release Date:</b> {row['release_date']}</p>
            <p><b>Genres:</b> {", ".join(row['genres'])}</p>
            <p><b>Matched:</b> {", ".join(row['matched_genres'])}</p>
            <p><b>Rating:</b> {row['vote_average']}</p>
            <p><b>Score:</b> {row['mood_score']}</p>
            <p>{row['overview']}</p>
        </div>
        """

    html = f"""
    <style>
        .header {{
            background: #1e3a8a;
            color: white;
            padding: 20px;
            border-radius: 12px;
            margin-bottom: 20px;
        }}
        .grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
            gap: 15px;
        }}
        .card {{
            background: white;
            border-left: 5px solid #1e3a8a;
            padding: 15px;
            border-radius: 12px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}
        .card h2 {{
            margin-top: 0;
            color: #1e3a8a;
        }}
    </style>

    <div class='header'>
        <h1>MSc Project: Mood-Based Movie Recommendation System</h1>
        <p>User Mood: {mood.capitalize()}</p>
    </div>

    <div class='grid'>
        {cards}
    </div>
    """

    display(HTML(html))
user_mood = input("Enter mood (happy, sad, excited, relaxed, romantic, scared): ")
results = recommend_movies(user_mood)
display(results)
show_html_results(results, user_mood)

Cleaned dataset shape: (97, 8)


,id,title,overview,genre_ids,release_date,vote_average,popularity,genres
0,1523145,Your Heart Will Be Broken,High school student Polina is saved from bully...,"[10749, 18]",2026-03-26,6.333,843.1812,"[Romance, Drama]"
1,1115544,Mike & Nick & Nick & Alice,Two gangsters and the woman they love try to s...,"[35, 878, 80]",2026-03-14,6.564,358.5571,"[Comedy, Science Fiction, Crime]"
2,1084187,Pretty Lethal,A troupe of ballerinas find themselves fightin...,"[28, 53, 27]",2026-03-13,6.800,308.4196,"[Action, Thriller, Horror]"
3,83533,Avatar: Fire and Ash,In the wake of the devastating war against the...,"[878, 12, 14]",2025-12-17,7.259,307.5829,"[Science Fiction, Adventure, Fantasy]"
4,687163,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[878, 12]",2026-03-15,8.153,307.2809,"[Science Fiction, Adventure]"


CSV saved successfully
Enter mood (happy, sad, excited, relaxed, romantic, scared): scared


,title,genres,matched_genres,overview,release_date,vote_average,mood_score
2,Pretty Lethal,"[Action, Thriller, Horror]","[Horror, Thriller]",A troupe of ballerinas find themselves fightin...,2026-03-13,6.800,12.484196
6,Send Help,"[Horror, Thriller, Comedy]","[Horror, Thriller]",Two colleagues become stranded on a deserted i...,2026-01-22,7.065,12.238426
9,Scream 7,"[Horror, Mystery, Crime]","[Horror, Mystery]",When a new Ghostface killer emerges in the qui...,2026-02-25,5.761,11.363809
29,The Housemaid,"[Mystery, Thriller]","[Mystery, Thriller]","Trying to escape her past, Millie Calloway acc...",2025-12-18,7.208,10.585097
62,Sinners,"[Horror, Action, Thriller]","[Horror, Thriller]","Trying to leave their troubled lives behind, t...",2025-04-16,7.469,10.298309
